## Demo for algos

In [1]:
# ======== ipynb settings =======
%load_ext autoreload
%autoreload 2

import os
import json
import torch
import numpy as np
import tempfile
import torchaudio
from src import InferenceAlgoRegistry

## Conditioned noise model



In [2]:
clean_path = "./data_demo/445c020m.wav"
noisy_path = "./data_demo/445c020m_STREET-KG-2_5.wav"


In [3]:
project_root = os.environ.get("PROJ_ROOT", os.getcwd())
metadata_jsonl = os.path.join(
    project_root, "sgmse/metadata_combination_encoded_audioldm_cpt/test.jsonl"
)
separate_speech_wsj0_ckpt_path = (
    os.environ.get("SPEECH_CKPT", "checkpoints/speech_prior.ckpt")
)
separate_noise_wsj0_ckpt_path = os.path.join(
    project_root, "logs/v2_01_08_2026/last.ckpt"
)
mix_audio_file = (
    os.environ.get("MIX_AUDIO_FILE", "data_demo/445c020m_STREET-KG-2_5.wav")
)
demo_output_dir = os.path.join(project_root, "demo_v2_11_08_2026")


In [4]:
for path, label in (
    (metadata_jsonl, "encoded training metadata"),
    (separate_speech_wsj0_ckpt_path, "speech checkpoint"),
    (separate_noise_wsj0_ckpt_path, "conditioned-noise checkpoint"),
    (mix_audio_file, "mixture audio"),
):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{label} not found: {path}")

os.makedirs(demo_output_dir, exist_ok=True)


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
def load_stored_embedding(jsonl_path, prompt, device):
    """Load the exact 512-D condition used to train this legacy checkpoint."""
    reference = None
    matches = 0

    with open(jsonl_path, "r", encoding="utf-8") as metadata_file:
        for line_number, line in enumerate(metadata_file, start=1):
            if not line.strip():
                continue

            item = json.loads(line)
            if str(item.get("text", "")).strip() != prompt.strip():
                continue
            if "embedding" not in item:
                raise ValueError(
                    f"Matching row {line_number} in {jsonl_path} has no embedding"
                )

            embedding = np.asarray(item["embedding"], dtype=np.float32)
            if embedding.shape != (512,):
                raise ValueError(
                    f"Expected a 512-D embedding at row {line_number}, "
                    f"got shape {embedding.shape}"
                )
            if not np.isfinite(embedding).all():
                raise ValueError(
                    f"Stored embedding at row {line_number} contains NaN or infinity"
                )

            if reference is None:
                reference = embedding
            elif not np.allclose(reference, embedding, rtol=0.0, atol=1e-7):
                raise ValueError(
                    f"Prompt {prompt!r} has inconsistent embeddings in {jsonl_path}"
                )
            matches += 1

    if reference is None:
        raise ValueError(
            f"Prompt {prompt!r} was not found in {jsonl_path}. This legacy "
            "checkpoint can only use prompts stored in its encoded training metadata."
        )

    print(
        f"Loaded stored condition for {prompt!r}: matches={matches}, "
        f"shape={reference.shape}, norm={np.linalg.norm(reference):.6f}"
    )
    return torch.from_numpy(reference).unsqueeze(0).to(device)


In [7]:
prompt_str = "This is street noise"
prompt_tensor = load_stored_embedding(
    metadata_jsonl,
    prompt_str,
    device,
)
print(f"Prompt tensor shape: {tuple(prompt_tensor.shape)}")

Loaded stored condition for 'This is street noise': matches=882, shape=(512,), norm=1.328808
Prompt tensor shape: (1, 512)


In [10]:
class IgnoreMissingTextCondition(torch.nn.Module):
    """Preserve the legacy speech model's unconditioned score path."""
    def forward(self, _condition):
        return 0.0


def make_legacy_checkpoint_loadable(checkpoint_path):
    """Adapt an old unconditioned speech checkpoint in a temporary copy."""
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state_dict = checkpoint.get("state_dict", {})
    hyper_parameters = checkpoint.get("hyper_parameters", {})
    nested = hyper_parameters.get("kwargs", {})
    if not isinstance(nested, dict):
        nested = {}

    # The current NCSN++ constructor always creates this additive projection,
    # while the legacy speech model predates text conditioning. Zero-valued
    # compatibility parameters satisfy strict checkpoint loading and add no
    # conditioning to the old speech score.
    weight_key = "dnn.text_embedding.weight"
    bias_key = "dnn.text_embedding.bias"
    if weight_key not in state_dict:
        # Infer 4*nf from an existing checkpoint tensor. This legacy speech
        # model uses nf=96 (output 384), while newer noise models use nf=128.
        timestep_bias = state_dict.get("dnn.all_modules.1.bias")
        if timestep_bias is not None:
            projection_dim = int(timestep_bias.numel())
        else:
            nf = int(hyper_parameters.get("nf", nested.get("nf", 96)))
            projection_dim = 4 * nf
        conditioning_dim = int(
            hyper_parameters.get(
                "conditioning_dim", nested.get("conditioning_dim", 512)
            )
        )
        state_dict[weight_key] = torch.zeros(projection_dim, conditioning_dim)
        state_dict[bias_key] = torch.zeros(projection_dim)
        print(
            "Added zero-effect text compatibility parameters to speech model: "
            f"weight shape={(projection_dim, conditioning_dim)}"
        )

    # Its EMA shadows also predate the two compatibility parameters. Removing
    # only the temporary copy's EMA lets Lightning restore the regular weights.
    checkpoint.pop("ema", None)
    handle, temporary_path = tempfile.mkstemp(
        prefix="audio2noise_legacy_speech_", suffix=".ckpt"
    )
    os.close(handle)
    torch.save(checkpoint, temporary_path)
    print(f"Using a temporary compatible checkpoint copy: {temporary_path}")
    return temporary_path, temporary_path


SeparateParaDiffUSEEN = InferenceAlgoRegistry.get_by_name("separate_paradiffuseen")

In [11]:
speech_checkpoint_for_loading, temporary_speech_checkpoint = (
    make_legacy_checkpoint_loadable(separate_speech_wsj0_ckpt_path)
)
try:
    separate_paradiffuseen = SeparateParaDiffUSEEN(
        ckpt_path=speech_checkpoint_for_loading,
        ckpt_noise=separate_noise_wsj0_ckpt_path,
        num_E=50,
        transform_type="exponent",
        verbose=True,
        listen_noise=True,
        device=str(device),
    )
    # separate_paradiffuseen passes None to the unconditioned speech prior.
    # Replace only that newly introduced projection; the noise prior keeps its
    # trained text-conditioning module and receives prompt_tensor below.
    separate_paradiffuseen.model.dnn.text_embedding = IgnoreMissingTextCondition()
    separate_paradiffuseen.model_cpu.dnn.text_embedding = IgnoreMissingTextCondition()
finally:
    if temporary_speech_checkpoint is not None:
        os.unlink(temporary_speech_checkpoint)
        temporary_speech_checkpoint = None

Added zero-effect text compatibility parameters to speech model: weight shape=(384, 512)


Lightning automatically upgraded your loaded checkpoint from v1.6.5 to v1.9.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../../../../tmp/audio2noise_legacy_speech_j5ubpe7m.ckpt`


Using a temporary compatible checkpoint copy: /tmp/audio2noise_legacy_speech_j5ubpe7m.ckpt


Lightning automatically upgraded your loaded checkpoint from v1.6.5 to v1.9.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../../../../tmp/audio2noise_legacy_speech_j5ubpe7m.ckpt`


In [12]:
# Set the seed after model construction for reproducible sampler noise.
torch.manual_seed(1234)
if device.type == "cuda":
    torch.cuda.manual_seed_all(1234)

In [13]:
assert prompt_tensor.shape == (1, 512)
assert torch.isfinite(prompt_tensor).all()
print(f"Condition source: {metadata_jsonl}")

Condition source: sgmse/metadata_combination_encoded_audioldm_cpt/test.jsonl


In [14]:
st, St, nt, Nt, st_post, St_post = separate_paradiffuseen.run(
        mix_file=mix_audio_file,
        clean_file=None,
        video_file=None,
        text_embedding=prompt_tensor, 
        nbatch=8,
        wiener_filter=True
    )

100%|███████████████████████████████████████████| 50/50 [01:23<00:00,  1.68s/it]


In [15]:
file_safe_prompt = prompt_str.replace(' ', '_')
    
speech_waveform = torch.tensor(st, dtype=torch.float32).unsqueeze(0).cpu()
noise_waveform = torch.tensor(nt, dtype=torch.float32).unsqueeze(0).cpu()

speech_out_path = os.path.join(demo_output_dir, f"separated_speech_{file_safe_prompt}.wav")
noise_out_path = os.path.join(demo_output_dir, f"separated_noise_{file_safe_prompt}.wav")

torchaudio.save(speech_out_path, speech_waveform, 16000)
torchaudio.save(noise_out_path, noise_waveform, 16000)
    
print(f"Audio separation artifacts written completely to:\n -> {speech_out_path}\n -> {noise_out_path}")

Audio separation artifacts written completely to:
 -> outputs/demo_v2/separated_speech_This_is_street_noise.wav
 -> outputs/demo_v2/separated_noise_This_is_street_noise.wav


## Conditioned noise model v3 (Conette captions)

In [23]:
v3_noise_checkpoint = os.path.join(
    project_root, "logs/v3_22_07_2026/last.ckpt"
)
v3_metadata_jsonl = os.path.join(
    project_root, "sgmse/conette_metadata_combination_encoded/test.jsonl"
)
v3_prompt = "a fan is blowing at a steady pace"
v3_mix_audio_file = mix_audio_file
v3_output_dir = os.path.join(project_root, "demo_v3_11_08_2026")

for path, label in (
    (v3_noise_checkpoint, "v3 noise checkpoint"),
    (v3_metadata_jsonl, "encoded v3 Conette metadata"),
    (v3_mix_audio_file, "v3 mixture audio"),
):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{label} not found: {path}")
os.makedirs(v3_output_dir, exist_ok=True)

In [24]:
def load_conette_condition(jsonl_path, prompt, device):
    """Load the exact stored v3 condition for a Conette caption."""
    normalized_prompt = str(prompt).strip()
    if not normalized_prompt:
        raise ValueError("v3_prompt must not be empty")

    selected_record = None
    with open(jsonl_path, "r", encoding="utf-8") as metadata_file:
        for line in metadata_file:
            if not line.strip():
                continue
            record = json.loads(line)
            record_caption = next(
                (
                    str(record[key]).strip()
                    for key in ("text", "caption", "description")
                    if record.get(key)
                ),
                None,
            )
            if record_caption == normalized_prompt:
                selected_record = record
                break

    if selected_record is None:
        raise ValueError(
            f"Conette caption {normalized_prompt!r} was not found in {jsonl_path}. "
            "For this checkpoint, v3_prompt must exactly match a stored caption."
        )
    if "embedding" not in selected_record:
        raise KeyError(f"Conette caption {normalized_prompt!r} has no embedding")

    embedding = np.asarray(selected_record["embedding"], dtype=np.float32).squeeze()
    if embedding.shape != (512,):
        raise ValueError(
            f"Expected a 512-D v3 embedding, got shape {embedding.shape}"
        )
    if not np.isfinite(embedding).all():
        raise ValueError("The selected v3 embedding contains NaN or infinity")

    caption = next(
        (
            str(selected_record[key]).strip()
            for key in ("text", "caption", "description")
            if selected_record.get(key)
        ),
        normalized_prompt,
    )
    wav_path = str(selected_record.get("wav_path", "unknown"))
    condition = torch.from_numpy(embedding).unsqueeze(0).to(device)
    return condition, caption, wav_path

In [25]:
v3_prompt_tensor, v3_caption, v3_condition_wav = load_conette_condition(
    v3_metadata_jsonl, v3_prompt, device
)
print(f"V3 caption: {v3_caption}")
print(f"V3 condition audio: {v3_condition_wav}")
print(f"V3 condition shape: {tuple(v3_prompt_tensor.shape)}")

V3 caption: a fan is blowing at a steady pace
V3 condition audio: data/text2noise/car/test/noise/example.wav
V3 condition shape: (1, 512)


In [26]:
speech_checkpoint_for_loading, temporary_speech_checkpoint = (
    make_legacy_checkpoint_loadable(separate_speech_wsj0_ckpt_path)
)
try:
    separate_paradiffuseen_v3 = SeparateParaDiffUSEEN(
        ckpt_path=speech_checkpoint_for_loading,
        ckpt_noise=v3_noise_checkpoint,
        num_E=50,
        transform_type="exponent",
        verbose=True,
        listen_noise=True,
        device=str(device),
    )
    separate_paradiffuseen_v3.model.dnn.text_embedding = (
        IgnoreMissingTextCondition()
    )
    separate_paradiffuseen_v3.model_cpu.dnn.text_embedding = (
        IgnoreMissingTextCondition()
    )
finally:
    if temporary_speech_checkpoint is not None:
        os.unlink(temporary_speech_checkpoint)
        temporary_speech_checkpoint = None

Added zero-effect text compatibility parameters to speech model: weight shape=(384, 512)


Lightning automatically upgraded your loaded checkpoint from v1.6.5 to v1.9.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../../../../tmp/audio2noise_legacy_speech_w0o_o0yg.ckpt`


Using a temporary compatible checkpoint copy: /tmp/audio2noise_legacy_speech_w0o_o0yg.ckpt


Lightning automatically upgraded your loaded checkpoint from v1.6.5 to v1.9.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../../../../../../../tmp/audio2noise_legacy_speech_w0o_o0yg.ckpt`


In [27]:
torch.manual_seed(1234)
if device.type == "cuda":
    torch.cuda.manual_seed_all(1234)

v3_st, v3_St, v3_nt, v3_Nt, v3_st_post, v3_St_post = (
    separate_paradiffuseen_v3.run(
        mix_file=v3_mix_audio_file,
        clean_file=None,
        video_file=None,
        text_embedding=v3_prompt_tensor,
        nbatch=8,
        wiener_filter=True,
    )
)

100%|███████████████████████████████████████████| 50/50 [01:08<00:00,  1.37s/it]


In [29]:
v3_output_name = "".join(
    character if character.isalnum() else "_" for character in v3_caption
).strip("_")[:80] or "conette_prompt"
v3_speech_path = os.path.join(
    v3_output_dir, f"separated_speech_{v3_output_name}.wav"
)
v3_noise_path = os.path.join(
    v3_output_dir, f"separated_noise_{v3_output_name}.wav"
)
torchaudio.save(
    v3_speech_path, torch.as_tensor(v3_st, dtype=torch.float32).unsqueeze(0).cpu(), 16000
)
torchaudio.save(
    v3_noise_path, torch.as_tensor(v3_nt, dtype=torch.float32).unsqueeze(0).cpu(), 16000
)
print(f"V3 separated speech: {v3_speech_path}")
print(f"V3 separated noise: {v3_noise_path}")

V3 separated speech: outputs/demo_v3/separated_speech_a_fan_is_blowing_at_a_steady_pace.wav
V3 separated noise: outputs/demo_v3/separated_noise_a_fan_is_blowing_at_a_steady_pace.wav


## Unconditioned noise model


In [ ]:
# Checkpoints path
separate_speech_wsj0_ckpt_path = os.environ.get("SPEECH_CKPT", "checkpoints/speech_prior.ckpt")
separate_noise_wsj0_ckpt_path = os.environ.get("NOISE_CKPT", "checkpoints/noise_prior.ckpt")

ckpt_av = "./ckpts/a_dummy_av_network.ckpt"

num_E = 30
verbose = True

SeparateParaDiffUSEEN = InferenceAlgoRegistry.get_by_name("separate_paradiffuseen")
separate_paradiffuseen = SeparateParaDiffUSEEN(ckpt_path=separate_speech_wsj0_ckpt_path, ckpt_noise =separate_noise_wsj0_ckpt_path , num_E=num_E, verbose=verbose, listen_noise=True)

                  

Lightning automatically upgraded your loaded checkpoint from v1.6.5 to v1.9.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file separate_wsjqut_speech_modeling.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.6.5 to v1.9.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file separate_wsjqut_speech_modeling.ckpt`


In [21]:
clean_path = "./data_demo/445c020m.wav"
noisy_path = "./data_demo/445c020m_STREET-KG-2_5.wav"

# clean_path_av = "./data_demo/clean_sa1_26M_Car_-5.wav"
# noisy_path_av = "./data_demo/noisy_sa1_26M_Car_-5.wav"
# v_path = "./data_demo/sa1_26M_mouthcrop.mp4"

In [10]:

# Run speech enhancement with paradiffuse
s_est, S_est, n_est, N_est, _, _= separate_paradiffuseen.run(
    mix_file=noisy_path, 
    clean_file=clean_path, 
    num_EM=1, 
    nbatch=4, 
    startstep=0,
    lmbd= 5.75,
)

# Show spectrograms and play audio
# show_spec(
#     spectogram=[separate_paradiffuseen.S_ref, separate_paradiffuseen.X.mean(0), S_est, N_est],
#     titles=["Clean", "Noisy", f"Estimated Speech", f"Estimated Noise"],
# )

print("Clean speech")
IPython.display.display(IPython.display.Audio(separate_paradiffuseen.s_ref, rate=16000))
print("Noisy speech")
IPython.display.display(IPython.display.Audio(separate_paradiffuseen.x, rate=16000))
print("Estimated speech separate_paradiffuseen")
IPython.display.display(IPython.display.Audio(s_est, rate=16000))
print("Estimated noise separate_paradiffuseen")
IPython.display.display(IPython.display.Audio(n_est, rate=16000))
           


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:13<00:00,  2.27it/s]

Clean speech


Noisy speech


Estimated speech separate_paradiffuseen


Estimated noise separate_paradiffuseen
